# seismogram atlas for MAVZ in 2022 

We compute a seismogram atlas for 3-component recordings from station MAVZ in 2022 following the methods described by [Steinmann et al., JGR, 2023](https://doi.org/10.1029/2023JB027167). We use 20 minute windows with 10 min overlap. 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from sklearn.decomposition import FastICA, PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
import mlflow
import xarray as xr
import pandas as pd
from zizou.visualise import plot_ssam_mpl
import umap

In [ ]:
fin_order1 = '/home/zizou/data/robust_forecasting/scatseisnet/order1.zarr'
fin_order2 = '/home/zizou/data/robust_forecasting/scatseisnet/order2.zarr'
order1 = xr.open_zarr(fin_order1, group='original')
order2 = xr.open_zarr(fin_order2, group='original')

In [ ]:
order1

In [ ]:
order2

In [ ]:
order2_stacked = order2.stack(freq_order2=('frequency', 'order2 frequency'))
order2_stacked

In [ ]:
frequency = order2.frequency.values
new_frequency = []
for i in range(0, len(order2['frequency'])-1):
    new_frequency.append(np.linspace(frequency[i], frequency[i+1],
                                     len(order2['order2 frequency']), endpoint=False))
new_frequency.append(np.linspace(frequency[-1], frequency[-1] - (frequency[-2] - frequency[-1]),
                                 len(order2['order2 frequency']), endpoint=False))
    
new_frequency = np.array(new_frequency).ravel()
order2_stacked = order2_stacked.assign_coords(freq_order2=new_frequency)
order2_stacked

In [ ]:
_ = plot_ssam_mpl(order1.order1.loc[dict(channel=0)].sel(frequency=slice(None, None, -1)).transpose('frequency', 'datetime'),
              dbscale=True, canvas_dim=None, log=True, cmap='magma')

In [ ]:
_ = plot_ssam_mpl(order2_stacked.order2.loc[dict(channel=0)].sel(freq_order2=slice(None, None, -1)).transpose('freq_order2', 'datetime'),
                  dbscale=True, canvas_dim=None, log=True, cmap='magma')

In [ ]:
o1 = order1.order1.stack(feature=('channel','frequency'))
o2 = order2_stacked.order2.rename({'freq_order2':'frequency'})
o2 = o2.stack(feature=('channel','frequency'))
combined = xr.concat([o1, o2], dim='feature')
X = combined.values

In [ ]:
pipe = Pipeline([("impute", SimpleImputer(strategy='median')),
                 ("center", StandardScaler(with_std=False)),
                 ("ica", FastICA(n_components=10, random_state=42))])
S = pipe.fit_transform(X)

In [ ]:
order = np.argsort(np.argmax(np.abs(S), axis=0))
xda_S = xr.DataArray(np.clip(S[:, order], -0.5, 0.5, out=S).T, dims=['sources', 'datetime'],
                     coords=[np.arange(S.shape[1]), combined.datetime])
#_ = plot_ssam_mpl(xda_S, dbscale=False, canvas_dim=None, cmap='RdBu_r')
xda_S.plot()

In [ ]:
reducer = umap.UMAP()
embedding = reducer.fit_transform(S)
embedding.shape

In [ ]:
# Plot the embedding as a 2D scatter plot colouring by the datetime in the original data
plt.figure(figsize=(10, 8))
dtnums = mdates.date2num(pd.to_datetime(combined.datetime.values))
plt.scatter(embedding[:, 0], embedding[:, 1], c=dtnums,
            cmap='magma', s=5, alpha=0.5)
cbar = plt.colorbar(label='Datetime')
# Show the datetime on the colorbar
locator = mdates.AutoDateLocator()
formatter = mdates.ConciseDateFormatter(locator)
cbar.ax.yaxis.set_major_locator(locator)
cbar.ax.yaxis.set_major_formatter(formatter)
cbar.ax.tick_params(labelsize=10)
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')
plt.title('UMAP Embedding of Scatseisnet Features')

### Comparison with unrest phases from Bramwell et al. [2025]

In [ ]:
def phase_plot(embedding, start, end, dtnums, ax, title):
    phase_indices = np.where((dtnums >= start) & (dtnums <= end))
    ax.scatter(embedding[:, 0], embedding[:, 1], c='gray',
                s=5, alpha=0.5)
    ax.scatter(embedding[phase_indices, 0], embedding[phase_indices, 1], c='purple',
                s=5, alpha=0.7)
    ax.set_xlabel('UMAP Component 1')
    ax.set_ylabel('UMAP Component 2')
    ax.set_title(title)

In [ ]:
phases = [ 
    (mdates.date2num(pd.to_datetime('2022-03-13')), mdates.date2num(pd.to_datetime('2022-03-17'))),
    (mdates.date2num(pd.to_datetime('2022-03-17')), mdates.date2num(pd.to_datetime('2022-03-23'))),
    (mdates.date2num(pd.to_datetime('2022-03-23')), mdates.date2num(pd.to_datetime('2022-03-24'))),
    (mdates.date2num(pd.to_datetime('2022-03-24')), mdates.date2num(pd.to_datetime('2022-04-05'))),
    (mdates.date2num(pd.to_datetime('2022-04-05')), mdates.date2num(pd.to_datetime('2022-04-09'))),
    (mdates.date2num(pd.to_datetime('2022-04-09')), mdates.date2num(pd.to_datetime('2022-04-12'))),
    (mdates.date2num(pd.to_datetime('2022-04-12')), mdates.date2num(pd.to_datetime('2022-05-08'))),
    (mdates.date2num(pd.to_datetime('2022-05-08')), mdates.date2num(pd.to_datetime('2022-06-19')))
]

fig, axs = plt.subplots(2, 4, figsize=(20, 10))
for i, (start, end) in enumerate(phases):
    ax = axs[i // 4, i % 4]
    phase_plot(embedding, start, end, dtnums, ax, f'Phase {i+1}')
